# 13 - Robot Dynamics And Energy

## What / Why / How

**What we are trying to do:** Simulate pendulum dynamics, energy, damping, and torque feedback control.

**Why this matters:** Dynamics explains why robots overshoot, oscillate, heat up, fall, or need torque-aware controllers.

**How we will do it:** Integrate a pendulum forward in time, compare free and controlled motion, and inspect energy loss and control effort.

## Goal

Dynamics asks: how do forces and torques create motion?

You will simulate:

- A pendulum
- Energy
- Damping
- A simple torque controller

In [ ]:
import math
import random
import sys
from pathlib import Path

import numpy as np

ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

try:
    import matplotlib.pyplot as plt
    HAS_PLOT = True
except ModuleNotFoundError:
    plt = None
    HAS_PLOT = False

np.set_printoptions(precision=3, suppress=True)

def plot_unavailable():
    if not HAS_PLOT:
        print("Install matplotlib to see plots: pip install -r requirements.txt")

In [ ]:
def simulate_pendulum(kp=0.0, kd=0.0, target=0.0, damping=0.05, dt=0.01, steps=1200):
    g, length, mass = 9.81, 1.0, 1.0
    theta, omega = np.deg2rad(80), 0.0
    rows = []
    for step in range(steps):
        error = target - theta
        torque = kp * error - kd * omega
        alpha = -(g / length) * np.sin(theta) - damping * omega + torque / (mass * length**2)
        omega += alpha * dt
        theta += omega * dt
        kinetic = 0.5 * mass * (length * omega) ** 2
        potential = mass * g * length * (1 - np.cos(theta))
        rows.append((step * dt, theta, omega, torque, kinetic + potential))
    return np.array(rows)

free = simulate_pendulum()
controlled = simulate_pendulum(kp=20.0, kd=5.0)
print("free final angle deg:", np.rad2deg(free[-1, 1]))
print("controlled final angle deg:", np.rad2deg(controlled[-1, 1]))

In [ ]:
if HAS_PLOT:
    plt.figure(figsize=(8, 3))
    plt.plot(free[:, 0], np.rad2deg(free[:, 1]), label="free")
    plt.plot(controlled[:, 0], np.rad2deg(controlled[:, 1]), label="controlled")
    plt.xlabel("time [s]")
    plt.ylabel("angle [deg]")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()
else:
    plot_unavailable()

In [ ]:
energy_drop = free[0, 4] - free[-1, 4]
control_effort = np.mean(np.abs(controlled[:, 3]))
print("energy lost to damping:", round(float(energy_drop), 3))
print("mean control effort:", round(float(control_effort), 3))

## Exercises

1. Set damping to zero and observe energy conservation.
2. Increase `kp` until the system oscillates.
3. Add torque saturation.
4. Explain why dynamics matters more for fast arms and legged robots.